In [0]:
import io, glob, hashlib, zipfile, requests
from datetime import datetime, timezone

ANO = 2026
URL = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/IPE/DADOS/ipe_cia_aberta_{ANO}.zip"
BASE = f"/Volumes/cvm/landing/indices/ipe/ano={ANO}"

def ingestao_existente(base, sha):
    """Procura uma ingestão anterior com o mesmo conteúdo."""
    for arquivo in glob.glob(f"{base}/ingestao=*/_sha256.txt"):
        with open(arquivo) as f:
            if f.read().strip() == sha:
                return arquivo.split("/")[-2]
    return None

r = requests.get(URL, timeout=120)
r.raise_for_status()
conteudo = io.BytesIO(r.content)
if not zipfile.is_zipfile(conteudo):
    raise ValueError("A resposta não é um ZIP válido:\n" + r.text[:300])

with zipfile.ZipFile(conteudo) as z:
    nomes = sorted(z.namelist())
    sha = hashlib.sha256(b"".join(z.read(n) for n in nomes)).hexdigest()

    existente = ingestao_existente(BASE, sha)
    if existente:
        DESTINO = f"{BASE}/{existente}"
        print(f"Conteúdo idêntico já ingerido em {existente}. Nada gravado.")
    else:
        ingestao = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        DESTINO = f"{BASE}/ingestao={ingestao}"
        z.extractall(DESTINO)
        with open(f"{DESTINO}/_sha256.txt", "w") as f:
            f.write(sha)
        print(f"Novo conteúdo gravado em {DESTINO}")

### Ler CSV

In [0]:
df = (spark.read
      .option("header", True)
      .option("sep", ";")
      .option("encoding", "ISO-8859-1")
      .csv(DESTINO))

df.printSchema()
print("Linhas:", df.count())
display(df.limit(20))

In [0]:
from pyspark.sql import functions as F

(df.groupBy("Categoria", "Tipo")
   .count()
   .orderBy(F.desc("count"))
   .show(40, truncate=False))

In [0]:
df.groupBy("Tipo_Apresentacao", "Versao").count().orderBy("Versao").show(truncate=False)

In [0]:
import requests

link = (df.filter(F.col("Categoria") == "Dados Econômico-Financeiros")
          .select("Link_Download").first()[0])
print(link)

r = requests.get(link, timeout=120)
print("Status:", r.status_code)
print("Tipo:", r.headers.get("Content-Type"))
print("Content-Disposition:", r.headers.get("Content-Disposition"))
print("Tamanho:", len(r.content), "bytes")
print("Primeiros bytes:", r.content[:8])

In [0]:
from pyspark.sql import functions as F

# Empresas da amostra, identificadas pelo Codigo_CVM (texto, como está no índice)
EMPRESAS_ALVO = {
    "9512":  "Petrobras",
    "4170":  "Vale",
    "19348": "Itaú Unibanco",
    "5410":  "WEG",
    "22470": "Magazine Luiza",
}

# Tipos de documento que interessam ao projeto
TIPOS_ALVO = [
    "Press-release",
    "Apresentações a analistas/agentes do mercado",
]

alvo = df.filter(
    F.col("Codigo_CVM").isin(list(EMPRESAS_ALVO)) &
    F.col("Tipo").isin(TIPOS_ALVO)
)
print("Documentos-alvo no índice:", alvo.count())

# Contagem por empresa, para conferir se bate com a tabela anterior (59 no total)
alvo.groupBy("Codigo_CVM", "Nome_Companhia", "Tipo").count() \
    .orderBy("Codigo_CVM", "Tipo").show(truncate=False)

# Grupos com mais de uma linha: são versões do mesmo documento ou documentos diferentes?
(alvo.groupBy("Codigo_CVM", "Tipo", "Data_Referencia", "Assunto")
     .agg(F.count("*").alias("linhas"),
          F.collect_list("Versao").alias("versoes"),
          F.collect_list("Data_Entrega").alias("entregas"))
     .filter("linhas > 1")
     .orderBy("Codigo_CVM", "Data_Referencia")
     .show(truncate=False))

In [0]:
# A) O que aconteceu com a versão 1 da Vale 4T25?
(df.filter((F.col("Codigo_CVM") == "4170") & (F.col("Data_Referencia") == "2025-12-31"))
   .select("Categoria", "Tipo", "Assunto", "Versao", "Data_Entrega", "Protocolo_Entrega")
   .orderBy("Versao")
   .show(truncate=False))

# B) Sem o Assunto, a chave confunde documentos diferentes?
(alvo.groupBy("Codigo_CVM", "Tipo", "Data_Referencia")
     .agg(F.count("*").alias("linhas"),
          F.collect_list("Versao").alias("versoes"),
          F.collect_list("Assunto").alias("assuntos"))
     .filter("linhas > 1")
     .orderBy("Codigo_CVM", "Data_Referencia")
     .show(truncate=False))


## Objetivo

Conhecer a fonte antes de desenhar o pipeline: o que o índice contém, como os
documentos são identificados e o que os links de download devolvem.

## Fonte

- **Conjunto:** Companhias Abertas — Documentos Periódicos e Eventuais (IPE), dados abertos da CVM
- **Índice:** `https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/IPE/DADOS/ipe_cia_aberta_{ANO}.zip`
- **Formato:** zip com um CSV, separador `;`, codificação `ISO-8859-1`
- **Volume (2026):** ~37 mil documentos listados, arquivo de ~1,7 MB
- **Documentos:** cada linha tem um `Link_Download` que entrega o PDF diretamente

## Estrutura criada

| Objeto | Função |
|---|---|
| `cvm.landing.indices` | Volume com os índices baixados, em `ipe/ano=AAAA/ingestao=<timestamp>/` |
| `cvm.landing.documentos` | Volume para os PDFs (usado a partir da Fase 2) |
| `cvm.bronze`, `cvm.silver`, `cvm.gold` | Schemas das camadas |

## Comportamento da fonte

| Observação | Evidência |
|---|---|
| Todas as colunas vêm como texto | `printSchema` |
| `Versao` é texto: ordena como `10, 11, ..., 2` | Contagem por versão |
| Nomes de empresa sem padrão (acentos, `SA` vs `S.A.`) e cortados em 50 caracteres | `ITAÚ`, `WEG SA`, `MAGAZINE LUIZA SA` |
| CNPJ genérico `00.000.000/0000-00` para empresas estrangeiras | JBS Foods International |
| ~8% dos documentos são reapresentações, até a versão 18 | Contagem por `Tipo_Apresentacao` e `Versao` |
| Há linhas `AP - Apresentação` com versão maior que 1 | Contagem por versão |
| Versões com buracos: o press-release 4T25 da Vale começa na versão 2 | Busca por todos os documentos da Vale em 31/12/2025 |
| Versões do mesmo documento podem ser entregues no mesmo dia | Vale (Investor Tour), Petrobras (1T26) |
| Protocolo muda a cada versão: não identifica o documento | JBS, Vale |
| Assunto com HTML escapado e pontuação variável | `Analyst &amp Investor Tour`, ponto final no 2T26 do Itaú |
| Mesma empresa, data e tipo podem ter documentos diferentes | WEG (apresentação, investidores, transcrição), Magalu (PT, EN, relatório anual) |
| Cada empresa classifica resultados de um jeito | Itaú usa `Relatório de Análise Gerencial` em vez de `Press-release` |
| O download declara `Content-Type: text/html`, mas o conteúdo é PDF | Primeiros bytes `%PDF-1.6` |

## Decisões

### Escopo

**Tipos de documento:**
- `Press-release`
- `Apresentações a analistas/agentes do mercado`

`Relatório de Análise Gerencial` (Itaú) ficou de fora por ser longo e caro de
processar. Pode ser incluído depois sem mudar código.

**Empresas (identificadas por `Codigo_CVM`):**

| Codigo_CVM | Empresa | Setor | Linhas no índice 2026 |
|---|---|---|---|
| 9512 | Petrobras | Energia | 9 |
| 4170 | Vale | Mineração | 19 |
| 19348 | Itaú Unibanco | Banco | 6 |
| 5410 | WEG | Indústria | 16 |
| 22470 | Magazine Luiza | Varejo | 9 |

Total: 59 linhas no índice, 50 documentos únicos após tratar versões.

### Identificação de empresas

Usar `Codigo_CVM`. Nome e CNPJ não são chaves confiáveis.

### Identificação de documentos

Chave: `Codigo_CVM + Tipo + Data_Referencia + Assunto normalizado`

- Sem o assunto, a chave junta documentos diferentes (WEG, Magalu).
- Com o assunto bruto, variações de digitação separariam versões do mesmo documento.
- **Normalização do assunto:** converter entidades HTML (`html.unescape`), remover
  espaços nas pontas e ponto final, passar para minúsculas.

### Versões

- Manter apenas a **maior versão** de cada documento.
- Converter `Versao` para número antes de comparar.
- Não supor que a versão 1 existe nem que as versões são contínuas.
- Desempatar pela versão, nunca pela `Data_Entrega`.

### Ingestão do índice

- Gravar cada download do índice numa pasta `ingestao=<timestamp>`, sem sobrescrever.
- Calcular o SHA-256 do CSV (não do zip) e gravar em `_sha256.txt`.
- Se o hash já existir, não gravar nada: a ingestão é idempotente.
- Validar que a resposta é um zip antes de extrair.

### Download de documentos

- Validar pelos primeiros bytes (`%PDF`), nunca pelo `Content-Type`.

### Idiomas

Releases em português e inglês (Magalu) são mantidos por enquanto. O idioma será
detectado pelo conteúdo após o parse, e a decisão de uso fica para o projeto 4.

## Achados para projetos futuros

- **Transcrições de teleconferências (WEG):** material rico para o assistente (projeto 4)
  e gabarito para a transcrição de vídeo (projeto 5).

## Próximos passos (Fase 1)

1. Tabela `bronze.ipe_indice` a partir dos CSVs da landing
2. Tabelas de controle com empresas e tipos-alvo
3. Tabela `silver.documentos_alvo` com a chave e a regra de versão acima